# 03 — Inference on a held-out test sample

Load the FM+phase+residual checkpoint, generate $E_z$ on a test geometry, and plot the
FDTD vs model triptych. The compliance percentage $\varepsilon_R$ (paper Eq. 9) is computed
with the same masked formula used in training.

**You need:** a checkpoint at `CKPT_PATH` and a dataset at `DATA_ROOT`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
for p in (REPO_ROOT, REPO_ROOT / 'Model', REPO_ROOT / 'tools'):
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)

DATA_ROOT = REPO_ROOT / 'Data' / 'unified_sweep_mmi_ybranch_dc_7500_each_1p55um'
CKPT_PATH = REPO_ROOT / 'checkpoints' / 'phase_300.pt'   # adjust to your checkpoint
print('data:', DATA_ROOT)
print('ckpt:', CKPT_PATH, '(exists)' if CKPT_PATH.is_file() else '(MISSING)')

In [ ]:
import json
import numpy as np
import torch

from predict_parametric_device import (
    _build_model_from_checkpoint,
    _build_cond_maps,
    _build_cond_vector,
    _checkpoint_state_dict,
    _ckpt_get,
)
from flow_matching import sample as fm_sample
from dataset import phase_anchor_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model  = _build_model_from_checkpoint(ckpt, device=device)
_, state = _checkpoint_state_dict(ckpt, use_ema=True)
model.load_state_dict(state, strict=True); model.eval()
print('model loaded; params =', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

In [ ]:
shards = DATA_ROOT / 'shards'
index  = json.loads((shards / 'index.json').read_text())
entry  = next(e for e in index
              if e['device'] == 'directional_coupler' and e.get('split') == 'test')
data   = np.load(shards / entry['shard'])
p      = f"s{entry['slot']}/"
eps    = np.asarray(data[p + 'eps'],     dtype=np.float32)
src    = np.asarray(data[p + 'src_mask'], dtype=np.float32)
ezr_gt = np.asarray(data[p + 'Ez_real'], dtype=np.float32)
ezi_gt = np.asarray(data[p + 'Ez_imag'], dtype=np.float32)
wavelength_um = float(np.asarray(data[p + 'wavelength_um']).item())
dx = float(np.asarray(data[p + 'dx_um']).item())
dy = float(np.asarray(data[p + 'dy_um']).item())
ezr_gt, ezi_gt, _ = phase_anchor_mask(ezr_gt, ezi_gt, src > 0.5, eps_r=eps, thr_eps=3.0)
print('geometry id:', entry.get('geometry_id'), '  shape:', eps.shape)

In [ ]:
stats     = ckpt['stats']
ckpt_args = ckpt.get('args')
cond_maps = _build_cond_maps(eps, src, stats=stats, ckpt_args=ckpt_args,
                              device=device, dx_um=dx, dy_um=dy)
cond      = _build_cond_vector(wavelength_um, stats, device=device)
lam_t     = torch.tensor([[wavelength_um]], device=device, dtype=torch.float32)

torch.manual_seed(0)
x0 = torch.randn((1, 2, eps.shape[0], eps.shape[1]), device=device, dtype=torch.float32)

with torch.no_grad():
    fields_norm = fm_sample(model, x0,
                            num_steps=100,
                            cond_maps=cond_maps, cond=cond, lambda_um=lam_t,
                            time_grid=str(_ckpt_get(ckpt_args, 'time_grid', 'linear')))
fn = fields_norm.float()
ezr = fn[0, 0].cpu().numpy() * float(stats['ez_real_std']) + float(stats['ez_real_mean'])
ezi = fn[0, 1].cpu().numpy() * float(stats['ez_imag_std']) + float(stats['ez_imag_mean'])

In [ ]:
from _triptych_metrics import residual_metrics

m = residual_metrics(eps, src, ezr_gt, ezi_gt, ezr, ezi,
                     dx_um=dx, dy_um=dy, wavelength_um=wavelength_um)
print(f"compliance eps_R = {m['eps_R_pct']:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(11, 3), constrained_layout=True)
ax[0].imshow(eps, origin='lower', cmap='viridis'); ax[0].set_title(r'$\varepsilon_r$')
ax[0].imshow(np.ma.masked_where(src <= 0.5, np.ones_like(src)),
             origin='lower', cmap='Greens', alpha=0.55, vmin=0, vmax=1)
vmax = float(np.percentile(np.abs(ezr_gt + 1j * ezi_gt), 99.5))
ax[1].imshow(np.abs(ezr_gt + 1j * ezi_gt), origin='lower', cmap='magma', vmin=0, vmax=vmax)
ax[1].set_title(r'FDTD $|E_z|$')
ax[2].imshow(np.abs(ezr + 1j * ezi),       origin='lower', cmap='magma', vmin=0, vmax=vmax)
ax[2].set_title(rf'PHASE $|E_z|$  ($\varepsilon_R={m["eps_R_pct"]:.1f}\%$)')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()